### 실습 1. 함수를 Runnable로 바꾸기 (API 키 불필요)

In [18]:
from langchain_core.runnables import RunnableLambda

def add_one(x):
    return x + 1

r_add = RunnableLambda(add_one)
print(r_add.invoke(3))
print(r_add.batch([1,2,3]))

4
[2, 3, 4]


### 실습 2. |로 연결하기 (API 키 불필요)

In [19]:
from langchain_core.runnables import RunnableLambda

# 1 = 2 동일하다
# 1. def add_one(x): return x + 1
# 2. 익명 함수 : lambda x: x+1
r_add = RunnableLambda(lambda x: x+1)
r_double = RunnableLambda(lambda x: x*2)

chain_a = r_add | r_double
chain_b = r_double | r_add

# chain 구조 확인
print(chain_a)
print(chain_b)

# invoke로 결과 확인
print(chain_a.invoke(2))
print(chain_b.invoke(3))

# Type 체크 : <class 'RunnableSequence'> 
print(type(chain_a))
print(type(chain_b))

first=RunnableLambda(lambda x: x + 1) middle=[] last=RunnableLambda(lambda x: x * 2)
first=RunnableLambda(lambda x: x * 2) middle=[] last=RunnableLambda(lambda x: x + 1)
6
7
<class 'langchain_core.runnables.base.RunnableSequence'>
<class 'langchain_core.runnables.base.RunnableSequence'>


### 실습 3. RunnableParallel과 RunnablePassthrough (API 키 불필요)

In [20]:
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough

# RunnableParallel : 병렬 처리(동시 처리) 
analyzer = RunnableParallel(
    original = RunnablePassthrough(),
    upper = RunnableLambda(lambda x: x.upper()),
    length = RunnableLambda(lambda x: len(x)),
)

print(analyzer.invoke("hello"))
# 결과 = {'original': 'hello', 'upper': 'HELLO', 'length': 5}
# dict 형식

{'original': 'hello', 'upper': 'HELLO', 'length': 5}


### 실습 4.  LLM 체인 만들기 (API 키 필요)

**부품을 하나씩 실행해 단계별 출력을 직접 확보세요.**

In [21]:
# langchain 패키지 설치 
%pip install -q langchain langchain-openai langchain-core python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

load_dotenv()  #.env 파일의  API키 불러오기

prompt = ChatPromptTemplate.from_template("{topic}에 대해서 친절한 초등학교 선생님처럼 설명해줘")

model = ChatOpenAI(model="gpt-4o-mini")
parser = StrOutputParser()

chain = prompt | model | parser

print(chain.invoke({"topic":"인공지능"}))

안녕하세요, 친구들! 오늘은 인공지능에 대해서 이야기해볼 거예요. 인공지능은 말 그대로 '인간처럼 생각하고 배우는 컴퓨터'를 뜻해요. 자, 그럼 어떻게 작동하는지 쉽게 설명해볼게요!

1. **배우는 기계**: 인공지능은 다양한 정보를 통해 배워요. 우리가 학교에서 공부하는 것처럼, 컴퓨터도 많은 데이터를 보고 패턴을 찾아내요. 예를 들어, 고양이와 개의 사진을 많이 보여주면, 고양이를 어떻게 알아보는지 배우게 되는 거죠.

2. **문제를 푸는 친구**: 인공지능은 우리처럼 문제를 해결할 수 있어요. 예를 들어, 우리가 수학 문제를 풀 때, 인공지능도 문제를 풀 수 있도록 만들어질 수 있어요. 그래서 게임을 하거나, 질문에 답할 때도 도움을 줄 수 있어요.

3. **대화하는 로봇**: 우리가 인공지능과 대화할 때, 마치 친구와 이야기하는 것처럼 느껴질 수 있어요. 인공지능은 사람의 말도 이해하고, 적절한 답변을 해줄 수 있게 프로그래밍 되어 있거든요. 그래서 우리가 물어보는 질문에 대답해줄 수 있는 거예요.

4. **일상 속의 인공지능**: 인공지능은 일상생활에서도 많이 사용돼요. 여러분이 사용하는 스마트폰이나, 자율주행차, 인터넷에서 추천해주는 영화나 음악도 인공지능 덕분이에요. 이처럼 우리 생활을 더 편리하게 만들어 주는 친구랍니다.

인공지능은 우리가 아니지만, 우리가 하는 것처럼 생각하고 행동할 수 있는 특별한 기술이에요. 앞으로 인공지능이 더 발전하면, 우리가 상상하지 못했던 멋진 일들도 배울 수 있을 거예요! 그래서 인공지능에 대해 더 많이 배우는 건 정말 신나고 중요하답니다. 

혹시 더 궁금한 점이 있나요?


### 실습 5. batch와 stream 써 보기 (API 키 필요)

**실습 4의 chain을 그대로 사용합니다.**

In [23]:
# 배치 처리
results = chain.batch([{"topic":"블록체인"},{"topic":"클라우드"}])

for i in results:
    print("-", i)

- 안녕하세요, 친구들! 오늘은 블록체인에 대해서 쉽고 재미있게 이야기해볼 거예요.

먼저, 블록체인이라는 것은 '블록'과 '체인'이 합쳐진 단어예요. '블록'은 정보를 담고 있는 데이터의 덩어리이고, '체인'은 그 블록들이 서로 연결되어 있는 것을 의미해요.

자, 이제 친구들이 이해하기 쉽게 비유를 들어볼게요. 한 마을에 모든 사람들이 함께 사용하는 큰 노트가 있다고 상상해보세요. 이 노트에는 마을 사람들 간의 거래나 중요한 정보가 기록돼요. 이 노트는 누구나 볼 수 있고, 아무나 마음대로 지울 수 없어요. 그래서 사람들이 서로를 믿고 거래할 수 있게 도와줘요.

이 큰 노트는 여러 페이지로 나뉘어져 있고, 각 페이지가 바로 '블록'이에요. 그리고 블록 하나가 추가될 때마다, 그 앞의 블록과 연결되어서 하나의 긴 '체인'이 형성돼요. 그래서 만약 어떤 사람이 쓴 내용이 잘못되면 이전 페이지를 확인할 수 있고, 다시 바꿀 수 없기 때문에 '속이기'가 아주 어렵습니다.

또한, 블록체인은 여러 사람이 동시에 보는 '공유된 기록'이기 때문에, 여러 명이 함께 확인하면서 안정성을 높일 수 있어요. 그래서 블록체인은 안전하고 믿을 수 있는 정보 기록 방법이라고 할 수 있어요.

블록체인은 주로 암호화폐라고 하는 디지털 돈이나 다른 중요한 정보를 저장하는 데 사용돼요. 예를 들어, 비트코인 같은 것이 바로 블록체인 기술을 이용해서 거래되는 것이죠.

정리하자면, 블록체인은 안전하고 변하지 않는 데이터를 기록하고 관리하는 방법이에요. 우리 함께 블록체인의 원리를 이해하고, 멋진 기술에 대해 더 알아보면 좋겠ですね! 질문이 있으면 언제든지 물어보세요!
- 안녕하세요, 친구들! 오늘은 "클라우드"에 대해서 이야기해볼 거예요. 클라우드는 우리가 인터넷을 통해 데이터를 저장하고 사용하는 방법 중 하나예요. 더 쉽게 설명해줄게요.

상상해보세요, 여러분이 좋아하는 그림이나 사진이 있다고 해볼게요. 이 그림을 여러분의 컴퓨터나 스마트폰에 저장할 수 있지만, 만약 그 기기가 고장이

In [25]:
# 스트림 처리
for chunk in chain.stream([{"topic":"블록체인"},{"topic":"클라우드"}]):
    print(chunk, end="", flush=True)

안녕하세요, 친구들! 오늘은 두 가지 재미있는 주제인 블록체인과 클라우드에 대해서 이야기해볼 거예요. 여러분 준비됐나요?

**블록체인**부터 시작할게요. 블록체인은 정보를 안전하게 저장하는 방법이에요. 우리가 흔히 쓰는 책이 있다고 생각해보세요. 책이 한 페이지에 여러 가지 내용이 적혀 있죠? 블록체인은 마치 그런 책의 페이지가 여러 개 연결된 것과 같아요. 하지만 여기서 중요한 점은, 각 페이지에는 이전 페이지와 연결되는 특별한 열쇠가 있어요. 그래서 한 페이지라도 변경하면 전체 책이 달라지게 되죠. 이렇게 해서 누군가가 정보를 마음대로 바꾸는 것을 막을 수 있어요. 블록체인은 사람들이 서로 신뢰할 수 있도록 도와주는 기술이에요!

이제 **클라우드**를 설명해볼게요. 클라우드는 구름처럼 하늘에 떠 있는 것처럼 보이지만, 사실은 인터넷을 통해 정보를 저장하고 접근할 수 있는 시스템이에요. 우리가 컴퓨터나 스마트폰에 저장한 파일들은 그 기기 안에만 들어있죠. 하지만 클라우드에 저장하면, 언제 어디서나 인터넷만 있으면 그 파일에 접근할 수 있어요. 마치 우리는 구름 위에서 하늘을 나는 새처럼, 언제든지 우리의 자료를 가져올 수 있는 거죠. 학교에서 친구들과 함께 작업할 때도 클라우드를 이용하면 서로의 자료를 쉽게 공유하고 협력할 수 있어요!

그래서 블록체인은 정보를 안전하게 저장하고, 클라우드는 그 정보를 언제 어디서나 쉽게 사용할 수 있도록 해주는 기술이에요. 이렇게 두 가지를 잘 활용하면, 우리 생활이 훨씬 편리해질 수 있어요! 궁금한 점이 있으면 언제든지 질문해 주세요!

### 도전과제

**과제 1 (기초, API 키 불필요) — 문자열을 받아 양쪽 공백을 없애고, 대문자로 바꾼 뒤, 끝에 !를 붙이는 체인을 만드세요.**

In [28]:
from langchain_core.runnables import RunnableLambda

def trimming(str):
    return str.strip().upper() + "!"

str_input = RunnableLambda(trimming)
print(str_input.invoke("  hello world  "))

HELLO WORLD!


**과제 2 (중급, API 키 불필요) — 숫자 리스트를 받아 합계, 평균, 최댓값을 한 번에 딕셔너리로 돌려주는 Runnable을 만드세요.**

In [43]:
from langchain_core.runnables import RunnableParallel, RunnableLambda

# 방법 1
def lists_summary(lists):
    return {"total" : sum(lists), "average" : sum(lists)/len(lists), "max" : max(lists)}

summary_runnable = RunnableLambda(lists_summary)
print(summary_runnable.invoke([10, 20, 30]))


# 방법 2
# RunnableParallel의 결과 Type = 딕셔너리
chain = RunnableParallel(
    total = RunnableLambda(lambda x:sum(x)),
    average = RunnableLambda(lambda x:sum(x)/len(x)),
    max = RunnableLambda(lambda x:max(x)),
)

print(chain.invoke([10, 20, 30]))

{'total': 60, 'average': 20.0, 'max': 30}
{'total': 60, 'average': 20.0, 'max': 30}


**과제 3 (심화, API 키 필요) — 주제 하나를 넣으면 ① 한 줄 설명과 ② 관련 키워드 3개를 동시에 만들어 딕셔너리로 돌려주는 체인을 만드세요.**

In [49]:
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

load_dotenv()  #.env 파일의  API키 불러오기

summary = ChatPromptTemplate.from_template("{topic}에 대해서 15년차 프로그램 개발자처럼 설명해줘. 분량은 1줄로 설명해줘.")
keywords = ChatPromptTemplate.from_template("{topic}에 대하여 15년차 프로그램 개발자의 입장에서 연관된 기술 용어 키워드 3개를 말해줘. 단어로만 말해줘")
model = ChatOpenAI(model="gpt-4o-mini", temperature=0.3, max_tokens=500)
str_parser = StrOutputParser()

summary_chain = summary | model | str_parser
keywords_chain = keywords | model | str_parser


print({"summary" : summary_chain.invoke({"topic":"머신러닝"}), "keywords" : keywords_chain.invoke({"topic":"머신러닝"})})

{'summary': '머신러닝은 데이터에서 패턴을 학습하여 예측이나 결정을 자동화하는 알고리즘 기반의 기술로, 다양한 분야에서 인사이트를 도출하고 문제를 해결하는 데 활용됩니다.', 'keywords': '딥러닝, 데이터 전처리, 모델 평가'}


In [48]:
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel
from langchain_openai import ChatOpenAI

load_dotenv()  # .env 파일의 API 키 불러오기

summary = ChatPromptTemplate.from_template(
    "{topic}에 대해서 15년차 프로그램 개발자처럼 설명해줘. 분량은 1줄로 설명해줘."
)
keywords = ChatPromptTemplate.from_template(
    "{topic}에 대하여 15년차 프로그램 개발자의 입장에서 연관된 기술 용어 키워드 3개를 말해줘. 단어로만 말해줘"
)
model = ChatOpenAI(model="gpt-4o-mini", temperature=0.3, max_tokens=500)
str_parser = StrOutputParser()

summary_chain = summary | model | str_parser
keywords_chain = keywords | model | str_parser

# 두 체인을 병렬로 실행하고 결과를 dict로 묶음
total_chain = RunnableParallel(
    summary = summary_chain,
    keywords = keywords_chain,
)

print(total_chain.invoke({"topic": "머신러닝"}))

{'summary': '머신러닝은 데이터에서 패턴을 학습하여 예측이나 결정을 자동으로 수행하는 알고리즘 기반의 기술로, 주로 통계학과 컴퓨터 과학의 융합으로 발전해왔습니다.', 'keywords': '딥러닝, 데이터 전처리, 모델 평가'}
